# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

- investigate how **chronic neuropathic pain changes the effects of long-term oxycodone use and withdrawal**
- compare **nerve-injured (SNI) mice** with **pain-free sham controls**
- examine both **behavioral withdrawal symptoms** and **gene-expression changes**
- analyze 3 brain regions involved in reward and addiction: **NAc, mPFC and VTA**
- identify molecular pathways and regulators linked to withdrawal, with **HDAC1/HDAC2** emerging as potential treatment targets

---

What do the conditions mean?

oxy: mice treated with oxycodone (treatment)

sal: mice treated with saline (control)

---

What do the genotypes mean?

*(experimental conditions, not genotypes)*

SNI (spared nerve injury): mice underwent nerve injury procedure to model chronic neuropathic pain

Sham (control surgery): mice underwent same procedure without damaging the nerves (pain-free controls)

---

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do?

- input: raw RNA-seq data
- perform quality control on the sequencing reads
- align reads to mouse reference genome
- generate gene-count matrix
- normalize counts 
- perform differential expression analysis
- analyze  brain regions (NAc, mPFC, VTA) separately
- use 2 × 2 design with factors SNI/Sham and Oxy/Sal (include interaction)

---

Which groups would you compare to each other? Please also mention which outcome you would expect to see from each comparison.

- **Sham-Oxy vs Sham-Sal**
  - Effect of **oxycodone withdrawal without neuropathic pain**
  - Expected: many withdrawal-associated DEGs; comparison produced **largest number of DEGs overall** in the study

- **SNI-Sal vs Sham-Sal**
  - Effect of **chronic neuropathic pain alone**
  - Expected: genes and pathways related to long-term effects of nerve injury/chronic pain

- **SNI-Oxy vs SNI-Sal**
  - Effect of **oxycodone withdrawal in animals with chronic pain**
  - Expected: withdrawal-related gene-expression changes that differ from those in pain-free mice

- **SNI-Oxy vs Sham-Sal**
  - Combined effect of **chronic pain + oxycodone withdrawal**.
  - Expected: strong, region-specific transcriptional changes; study found many DEGs in all 3 brain regions

- **Interaction: SNI × Oxycodone**
  - Tests whether effect of oxycodone withdrawal **depends on whether chronic pain is present**
  - Expected: significant interaction for genes whose withdrawal response is different in SNI and Sham mice—the main biological question of the study

---

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [1]:
# convert table into tidy format:
# 1 row per sequencing run 
# separate columns for Run, Condition, and Genotype
# remove original x/empty indicator columns 

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("conditions_runs_oxy_project.xlsx")

# Create a cleaner table
df_clean = pd.DataFrame({
    "Run": df["Run"],
    "Condition": df["Condition: Oxy"].notna().map({True: "Oxy", False: "Sal"}),
    "Genotype": df["Genotype: SNI"].notna().map({True: "SNI", False: "Sham"})
})

# Sort samples
df_clean = df_clean.sort_values(
    ["Genotype", "Condition", "Run"]
).reset_index(drop=True)

df_clean

,Run,Condition,Genotype
0,SRR23195508,Oxy,SNI
1,SRR23195509,Oxy,SNI
2,SRR23195516,Oxy,SNI
3,SRR23195517,Oxy,SNI
4,SRR23195505,Sal,SNI
5,SRR23195510,Sal,SNI
6,SRR23195513,Sal,SNI
7,SRR23195518,Sal,SNI
8,SRR23195506,Oxy,Sham
9,SRR23195511,Oxy,Sham


In [2]:
# 1. Samples per condition
condition_counts = df_clean["Condition"].value_counts()
print(condition_counts)

# 2. Samples per genotype
genotype_counts = df_clean["Genotype"].value_counts()
print(genotype_counts)

# 3. Conditions per genotype
condition_genotype = pd.crosstab(
    df_clean["Genotype"],
    df_clean["Condition"]
)
print(condition_genotype)

Condition
Oxy    8
Sal    8
Name: count, dtype: int64
Genotype
SNI     8
Sham    8
Name: count, dtype: int64
Condition  Oxy  Sal
Genotype           
SNI          4    4
Sham         4    4


In [4]:
from pathlib import Path

img_dir = Path("day_02_img")
img_dir.mkdir(exist_ok=True)

condition_genotype.plot(kind="bar")
plt.xlabel("Genotype")
plt.ylabel("Number of samples")
plt.title("Condition per genotype")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig("day_02_img/condition_per_genotype.png", dpi=300)
plt.close()

(completely balanced) dataset contains 16 RNA-seq samples:
- samples per condition: Oxy = 8, Sal = 8
- samples per genotype: SNI = 8, Sham = 8
- condition per genotype:
  - SNI-Oxy = 4
  - SNI-Sal = 4
  - Sham-Oxy = 4
  - Sham-Sal = 4

<div align="center">
<img src="day_02_img/condition_per_genotype.png" width="350">
</div>

---

They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.